# Creation of SOLAQUA labeled fish

## Getting fish images

In [1]:
from pathlib import Path
import shutil
import csv

# -----------------------------------------------------------------------------
# Paths
# -----------------------------------------------------------------------------
PROJECT_ROOT = Path()
ALL_IMAGES_ROOT = PROJECT_ROOT / "raw_processed" / "all_images"
DATASET_ROOT = PROJECT_ROOT / "solaqua_labeled_fish"
DEST_DIR = DATASET_ROOT / "images"
MANIFEST_PATH = DATASET_ROOT / "selected_images_manifest.csv"

DEST_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------------------------------------------------------
# Config
# -----------------------------------------------------------------------------
# Copy every Nth selected frame.
# - 1  -> copy all selected frames
# - 5  -> copy every 5th selected frame
# - 10 -> good if you want a smaller manual labeling set
SAMPLE_EVERY_N = 4

# If False, existing files in DEST_DIR are left untouched and skipped.
OVERWRITE = False

# If True, only prints what would be copied.
DRY_RUN = False

VALID_EXTS = {".jpg", ".jpeg", ".png"}

if SAMPLE_EVERY_N < 1:
    raise ValueError("SAMPLE_EVERY_N must be >= 1")

# -----------------------------------------------------------------------------
# Bag definitions
# Notes:
# - A single note like "3s" is encoded as (3, 4), meaning the 3rd second.
# - "NO FISH" bags are included with empty intervals and will be skipped.
# -----------------------------------------------------------------------------
BAGS = [
    {"bag_id": "bag1",  "bag_name": "2024-08-20_13-55-34", "intervals": [(2.5, 4), (30, 40)]},
    {"bag_id": "bag2",  "bag_name": "2024-08-20_13-57-42", "intervals": [(7, 13), (29, 35)]},
    {"bag_id": "bag3",  "bag_name": "2024-08-20_14-16-05", "intervals": [(43, 46)]},
    {"bag_id": "bag4",  "bag_name": "2024-08-20_14-31-29", "intervals": [(53, 67)]},
    {"bag_id": "bag5",  "bag_name": "2024-08-20_14-34-07", "intervals": []},
    {"bag_id": "bag6",  "bag_name": "2024-08-20_14-36-22", "intervals": []},
    {"bag_id": "bag7",  "bag_name": "2024-08-20_14-38-37", "intervals": []},
    {"bag_id": "bag8",  "bag_name": "2024-08-20_15-20-29", "intervals": [(39, 41), (46, 49)]},
    {"bag_id": "bag9",  "bag_name": "2024-08-20_16-34-34", "intervals": [(92, 95)]},
    {"bag_id": "bag10", "bag_name": "2024-08-20_16-43-25", "intervals": [(48, 49), (60, 69)]},
    {"bag_id": "bag11", "bag_name": "2024-08-20_16-45-21", "intervals": [(37, 47)]},
    {"bag_id": "bag12", "bag_name": "2024-08-20_16-47-54", "intervals": [(19, 27), (69, 76)]},
    {"bag_id": "bag13", "bag_name": "2024-08-20_17-02-00", "intervals": [(41, 44), (47, 60)]},
    {"bag_id": "bag14", "bag_name": "2024-08-20_17-14-36", "intervals": [(0, 8), (22, 25), (27, 41)]},
    {"bag_id": "bag15", "bag_name": "2024-08-20_17-55-40", "intervals": [(1, 9), (29, 36), (43, 50)]},
    {"bag_id": "bag16", "bag_name": "2024-08-20_18-47-40", "intervals": []},
    {"bag_id": "bag17", "bag_name": "2024-08-20_18-50-22", "intervals": []},
    {"bag_id": "bag18", "bag_name": "2024-08-20_18-53-59", "intervals": []},
]

# -----------------------------------------------------------------------------
# Helpers
# -----------------------------------------------------------------------------
def load_timestamped_images(folder: Path):
    """
    Return [(timestamp_ns, path), ...] sorted by timestamp.
    Assumes filenames are like 1724166879127906200.jpg
    """
    items = []
    for p in folder.iterdir():
        if not p.is_file():
            continue
        if p.suffix.lower() not in VALID_EXTS:
            continue
        try:
            ts_ns = int(p.stem)
        except ValueError:
            continue
        items.append((ts_ns, p))

    items.sort(key=lambda x: x[0])
    return items


def is_in_intervals(rel_s: float, intervals):
    return any(start <= rel_s <= end for start, end in intervals)


# -----------------------------------------------------------------------------
# Main
# -----------------------------------------------------------------------------
manifest_rows = []
total_selected = 0
total_copied = 0
total_skipped_existing = 0

for bag in BAGS:
    bag_id = bag["bag_id"]
    bag_name = bag["bag_name"]
    intervals = bag["intervals"]

    if not intervals:
        print(f"[SKIP] {bag_id} ({bag_name}): no fish intervals")
        continue

    src_dir = ALL_IMAGES_ROOT / bag_name
    if not src_dir.exists():
        print(f"[WARN] {bag_id} ({bag_name}): missing source dir -> {src_dir}")
        continue

    images = load_timestamped_images(src_dir)
    if not images:
        print(f"[WARN] {bag_id} ({bag_name}): no timestamped images found")
        continue

    first_ts_ns = images[0][0]

    # Select frames inside your fish windows
    selected = []
    for ts_ns, src_path in images:
        rel_s = (ts_ns - first_ts_ns) / 1_000_000_000
        if is_in_intervals(rel_s, intervals):
            selected.append((ts_ns, rel_s, src_path))

    # Optional subsampling
    sampled = selected[::SAMPLE_EVERY_N]

    bag_copied = 0
    bag_skipped_existing = 0

    for ts_ns, rel_s, src_path in sampled:
        dst_name = f"{bag_id}_{ts_ns}{src_path.suffix.lower()}"
        dst_path = DEST_DIR / dst_name

        if dst_path.exists() and not OVERWRITE:
            bag_skipped_existing += 1
            total_skipped_existing += 1
            continue

        if not DRY_RUN:
            shutil.copy2(src_path, dst_path)

        manifest_rows.append({
            "bag_id": bag_id,
            "bag_name": bag_name,
            "original_timestamp_ns": ts_ns,
            "relative_time_s": f"{rel_s:.3f}",
            "source_path": str(src_path),
            "dest_path": str(dst_path),
        })

        bag_copied += 1
        total_copied += 1

    total_selected += len(sampled)

    print(
        f"[DONE] {bag_id} ({bag_name}) | "
        f"source images: {len(images)} | "
        f"selected: {len(selected)} | "
        f"after sampling: {len(sampled)} | "
        f"copied: {bag_copied} | "
        f"skipped existing: {bag_skipped_existing}"
    )

# -----------------------------------------------------------------------------
# Write manifest
# -----------------------------------------------------------------------------
if manifest_rows:
    with MANIFEST_PATH.open("w", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=[
                "bag_id",
                "bag_name",
                "original_timestamp_ns",
                "relative_time_s",
                "source_path",
                "dest_path",
            ],
        )
        writer.writeheader()
        writer.writerows(manifest_rows)

print("\n[SUMMARY]")
print(f"Selected after sampling: {total_selected}")
print(f"Copied: {total_copied}")
print(f"Skipped existing: {total_skipped_existing}")
print(f"Images folder: {DEST_DIR}")
print(f"Manifest: {MANIFEST_PATH}")
print(f"DRY_RUN={DRY_RUN}, SAMPLE_EVERY_N={SAMPLE_EVERY_N}, OVERWRITE={OVERWRITE}")

[DONE] bag1 (2024-08-20_13-55-34) | source images: 1230 | selected: 287 | after sampling: 72 | copied: 72 | skipped existing: 0
[DONE] bag2 (2024-08-20_13-57-42) | source images: 1997 | selected: 301 | after sampling: 76 | copied: 76 | skipped existing: 0
[DONE] bag3 (2024-08-20_14-16-05) | source images: 1308 | selected: 75 | after sampling: 19 | copied: 19 | skipped existing: 0
[DONE] bag4 (2024-08-20_14-31-29) | source images: 1688 | selected: 350 | after sampling: 88 | copied: 88 | skipped existing: 0
[SKIP] bag5 (2024-08-20_14-34-07): no fish intervals
[SKIP] bag6 (2024-08-20_14-36-22): no fish intervals
[SKIP] bag7 (2024-08-20_14-38-37): no fish intervals
[DONE] bag8 (2024-08-20_15-20-29) | source images: 1476 | selected: 126 | after sampling: 32 | copied: 32 | skipped existing: 0
[DONE] bag9 (2024-08-20_16-34-34) | source images: 2377 | selected: 74 | after sampling: 19 | copied: 19 | skipped existing: 0
[DONE] bag10 (2024-08-20_16-43-25) | source images: 1723 | selected: 247 | 